### Prompt engineering
Is the process of taking an initial prompt and improving it to get more reliable, higher-quality outputs
- set a goal
- write an initial prompt: basic first attempt
- evaluate the prompt: test against the criteria set in goals
- apply prompt engineering techniques: use methods to improve on performance
- evaluate the improved prompt for even better performance: verify that the changes improved on the results

Example:
Goal is to create a prompt that generates one day meal plans for athletes
- taking into account:
    - height
    - weight
    - goals (that they want to accomplish following the meal plan from the prompt)
    - dietary restrictions

In [24]:
from dotenv import load_dotenv

load_dotenv()

from anthropic import Anthropic
client = Anthropic()
model = "claude-haiku-4-5"

def add_messages(messages, text, role="user"):
    message = {"role": role, "content": text}
    messages.append(message)

def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [25]:
# Report Builder
from statistics import mean

def generate_prompt_evaluation_report(evaluation_results):
    total_tests = len(evaluation_results)
    scores = [result["score"] for result in evaluation_results]
    avg_score = mean(scores) if scores else 0
    max_possible_score = 10
    pass_rate = (
        100 * len([s for s in scores if s >= 7]) / total_tests if total_tests else 0
    )

    html = f"""
    <!DOCTYPE html>
    <html lang="en">
    <head>
        <meta charset="UTF-8">
        <meta name="viewport" content="width=device-width, initial-scale=1.0">
        <title>Prompt Evaluation Report</title>
        <style>
            body {{
                font-family: Arial, sans-serif;
                line-height: 1.6;
                margin: 0;
                padding: 20px;
                color: #333;
            }}
            .header {{
                background-color: #f0f0f0;
                padding: 20px;
                border-radius: 5px;
                margin-bottom: 20px;
            }}
            .summary-stats {{
                display: flex;
                justify-content: space-between;
                flex-wrap: wrap;
                gap: 10px;
            }}
            .stat-box {{
                background-color: #fff;
                border-radius: 5px;
                padding: 15px;
                box-shadow: 0 2px 5px rgba(0,0,0,0.1);
                flex-basis: 30%;
                min-width: 200px;
            }}
            .stat-value {{
                font-size: 24px;
                font-weight: bold;
                margin-top: 5px;
            }}
            table {{
                width: 100%;
                border-collapse: collapse;
                margin-top: 20px;
            }}
            th {{
                background-color: #4a4a4a;
                color: white;
                text-align: left;
                padding: 12px;
            }}
            td {{
                padding: 10px;
                border-bottom: 1px solid #ddd;
                vertical-align: top;
            }}
            tr:nth-child(even) {{
                background-color: #f9f9f9;
            }}
            .output-cell {{
                white-space: pre-wrap;
            }}
            .score {{
                font-weight: bold;
                padding: 5px 10px;
                border-radius: 3px;
                display: inline-block;
            }}
            .score-high {{
                background-color: #c8e6c9;
                color: #2e7d32;
            }}
            .score-medium {{
                background-color: #fff9c4;
                color: #f57f17;
            }}
            .score-low {{
                background-color: #ffcdd2;
                color: #c62828;
            }}
            .output {{
                overflow: auto;
                white-space: pre-wrap;
            }}

            .output pre {{
                background-color: #f5f5f5;
                border: 1px solid #ddd;
                border-radius: 4px;
                padding: 10px;
                margin: 0;
                font-family: 'Consolas', 'Monaco', 'Courier New', monospace;
                font-size: 14px;
                line-height: 1.4;
                color: #333;
                box-shadow: inset 0 1px 3px rgba(0, 0, 0, 0.1);
                overflow-x: auto;
                white-space: pre-wrap; 
                word-wrap: break-word; 
            }}

            td {{
                width: 20%;
            }}
            .score-col {{
                width: 80px;
            }}
        </style>
    </head>
    <body>
        <div class="header">
            <h1>Prompt Evaluation Report</h1>
            <div class="summary-stats">
                <div class="stat-box">
                    <div>Total Test Cases</div>
                    <div class="stat-value">{total_tests}</div>
                </div>
                <div class="stat-box">
                    <div>Average Score</div>
                    <div class="stat-value">{avg_score:.1f} / {max_possible_score}</div>
                </div>
                <div class="stat-box">
                    <div>Pass Rate (≥7)</div>
                    <div class="stat-value">{pass_rate:.1f}%</div>
                </div>
            </div>
        </div>

        <table>
            <thead>
                <tr>
                    <th>Scenario</th>
                    <th>Prompt Inputs</th>
                    <th>Solution Criteria</th>
                    <th>Output</th>
                    <th>Score</th>
                    <th>Reasoning</th>
                </tr>
            </thead>
            <tbody>
    """

    for result in evaluation_results:
        prompt_inputs_html = "<br>".join(
            [
                f"<strong>{key}:</strong> {value}"
                for key, value in result["test_case"]["prompt_inputs"].items()
            ]
        )

        criteria_string = "<br>• ".join(result["test_case"]["solution_criteria"])

        score = result["score"]
        if score >= 8:
            score_class = "score-high"
        elif score <= 5:
            score_class = "score-low"
        else:
            score_class = "score-medium"

        html += f"""
            <tr>
                <td>{result["test_case"]["scenario"]}</td>
                <td class="prompt-inputs">{prompt_inputs_html}</td>
                <td class="criteria">• {criteria_string}</td>
                <td class="output"><pre>{result["output"]}</pre></td>
                <td class="score-col"><span class="score {score_class}">{score}</span></td>
                <td class="reasoning">{result["reasoning"]}</td>
            </tr>
        """

    html += """
            </tbody>
        </table>
    </body>
    </html>
    """

    return html


In [26]:
# PromptEvaluator Implementation
import re
from textwrap import dedent
import json
import concurrent.futures

class PromptEvaluator:
    def __init__(self, max_concurrent_tasks=3):
        self.max_concurrent_tasks = max_concurrent_tasks

    def render(self, template_string, variables):
        placeholders = re.findall(r"{([^{}]+)}", template_string)

        result = template_string
        for placeholder in placeholders:
            if placeholder in variables:
                result = result.replace(
                    "{" + placeholder + "}", str(variables[placeholder])
                )

        return result.replace("{{", "{").replace("}}", "}")

    def generate_unique_ideas(self, task_description, prompt_inputs_spec, num_cases):
        """Generate a list of unique ideas for test cases based on the task description"""

        prompt = """
        Generate {num_cases} unique, diverse ideas for testing a prompt that accomplishes this task:
        
        <task_description>
        {task_description}
        </task_description>

        The prompt will receive the following inputs
        <prompt_inputs>
        {prompt_inputs_spec}
        </prompt_inputs>
        
        Each idea should represent a distinct scenario or example that tests different aspects of the task.
        
        Output Format:
        Provide your response as a structured JSON array where each item is a brief description of the idea.
        
        Example:
        ```json
        [
            "Testing with technical computer science terminology",
            "Testing with medical research findings",
            "Testing with complex mathematical concepts",
            ...
        ]
        ```
        
        Ensure each idea is:
        - Clearly distinct from the others
        - Relevant to the task description
        - Specific enough to guide generation of a full test case
        - Quick to solve without requiring extensive computation or multi-step processing
        - Solvable with no more than 400 tokens of output

        Remember, only generate {num_cases} unique ideas
        """

        system_prompt = "You are a test scenario designer specialized in creating diverse, unique testing scenarios."

        example_prompt_inputs = ""
        for key, value in prompt_inputs_spec.items():
            val = value.replace("\n", "\\n")
            example_prompt_inputs += f'"{key}": str # {val},'

        rendered_prompt = self.render(
            dedent(prompt),
            {
                "task_description": task_description,
                "num_cases": num_cases,
                "prompt_inputs": example_prompt_inputs,
            },
        )

        messages = []
        add_messages(messages, rendered_prompt, role="user")
        add_messages(messages, "```json", role="assistant")
        text = chat(
            messages,
            stop_sequences=["```"],
            system=system_prompt,
            temperature=1.0,
        )

        return json.loads(text)

    def generate_test_case(self, task_description, idea, prompt_inputs_spec={}):
        """Generate a single test case based on the task description and a specific idea"""

        example_prompt_inputs = ""
        for key, value in prompt_inputs_spec.items():
            val = value.replace("\n", "\\n")
            example_prompt_inputs += f'"{key}": "EXAMPLE_VALUE", // {val}\n'

        allowed_keys = ", ".join([f'"{key}"' for key in prompt_inputs_spec.keys()])

        prompt = """
        Generate a single detailed test case for a prompt evaluation based on:
        
        <task_description>
        {task_description}
        </task_description>
        
        <specific_idea>
        {idea}
        </specific_idea>
        
        <allowed_input_keys>
        {allowed_keys}
        </allowed_input_keys>
        
        Output Format:
        ```json
        {{
            "prompt_inputs": {{
            {example_prompt_inputs}
            }},
            "solution_criteria": ["criterion 1", "criterion 2", ...] // Concise list of criteria for evaluating the solution, 1 to 4 items
        }}
        ```
        
        IMPORTANT REQUIREMENTS:
        - You MUST ONLY use these exact input keys in your prompt_inputs: {allowed_keys}        
        - Do NOT add any additional keys to prompt_inputs
        - All keys listed in allowed_input_keys must be included in your response
        - Make the test case realistic and practically useful
        - Include measurable, concise solution criteria
        - The solution criteria should ONLY address the direct requirements of the task description and the generated prompt_inputs
        - Avoid over-specifying criteria with requirements that go beyond the core task
        - Keep solution criteria simple, focused, and directly tied to the fundamental task
        - The test case should be tailored to the specific idea provided
        - Quick to solve without requiring extensive computation or multi-step processing
        - Solvable with no more than 400 tokens of output
        - DO NOT include any fields beyond those specified in the output format

        Here's an example of a sample input with an ideal output:
        <sample_input>
        <sample_task_description>
        Extract topics out of a passage of text
        </sample_task_description>
        <sample_specific_idea>
        Testing with a text that contains multiple nested topics and subtopics (e.g., a passage about renewable energy that covers solar power economics, wind turbine technology, and policy implications simultaneously)
        </sample_specific_idea>

        <sample_allowed_input_keys>
        "content"
        </sample_allowed_input_keys>
        </sample_input>
        <ideal_output>
        ```json
        {
            "prompt_inputs": {
                "content": "The transition to renewable energy encompasses numerous interdependent dimensions. Solar photovoltaic technology has seen dramatic cost reductions, with panel efficiency improving 24% since 2010 while manufacturing costs declined by 89%, making it economically competitive with fossil fuels in many markets. Concurrently, wind energy has evolved through innovative turbine designs featuring carbon-fiber composite blades and advanced control systems that increase energy capture by 35% in low-wind conditions."
            },
            "solution_criteria": [
                "Includes all topics mentioned"   
            ]
        }
        ```
        </ideal_output>
        This is ideal output because the solution criteria is concise and doesn't ask for anything outside of the scope of the task description.
        """

        system_prompt = "You are a test case creator specializing in designing evaluation scenarios."

        rendered_prompt = self.render(
            dedent(prompt),
            {
                "allowed_keys": allowed_keys,
                "task_description": task_description,
                "idea": idea,
                "example_prompt_inputs": example_prompt_inputs,
            },
        )

        messages = []
        add_messages(messages, rendered_prompt, role="user")
        add_messages(messages, "```json", role="assistant")
        text = chat(
            messages,
            stop_sequences=["```"],
            system=system_prompt,
            temperature=0.7,
        )

        test_case = json.loads(text)
        test_case["task_description"] = task_description
        test_case["scenario"] = idea

        return test_case

    def generate_dataset(
        self,
        task_description,
        prompt_inputs_spec={},
        num_cases=1,
        output_file="dataset.json",
    ):
        """Generate test dataset based on task description and save to file"""
        ideas = self.generate_unique_ideas(
            task_description, prompt_inputs_spec, num_cases
        )

        dataset = []
        completed = 0
        total = len(ideas)
        last_reported_percentage = 0

        with concurrent.futures.ThreadPoolExecutor(
            max_workers=self.max_concurrent_tasks
        ) as executor:
            future_to_idea = {
                executor.submit(
                    self.generate_test_case,
                    task_description,
                    idea,
                    prompt_inputs_spec,
                ): idea
                for idea in ideas
            }

            for future in concurrent.futures.as_completed(future_to_idea):
                try:
                    result = future.result()
                    completed += 1
                    current_percentage = int((completed / total) * 100)
                    milestone_percentage = (current_percentage // 20) * 20

                    if milestone_percentage > last_reported_percentage:
                        print(f"Generated {completed}/{total} test cases")
                        last_reported_percentage = milestone_percentage

                    dataset.append(result)
                except Exception as e:
                    print(f"Error generating test case: {e}")

        with open(output_file, "w") as f:
            json.dump(dataset, f, indent=2)

        return dataset

    def grade_output(self, test_case, output, extra_criteria):
        """Grade the output of a test case using the model"""

        prompt_inputs = ""
        for key, value in test_case["prompt_inputs"].items():
            val = value.replace("\n", "\\n")
            prompt_inputs += f'"{key}":"{val}",\n'

        extra_criteria_section = ""
        if extra_criteria:
            extra_criteria_template = """
            Mandatory Requirements - ANY VIOLATION MEANS AUTOMATIC FAILURE (score of 3 or lower):
            <extra_important_criteria>
            {extra_criteria}
            </extra_important_criteria>
            """
            extra_criteria_section = self.render(
                dedent(extra_criteria_template),
                {"extra_criteria": extra_criteria},
            )

        eval_template = """
        Your task is to evaluate the following AI-generated solution with EXTREME RIGOR.

        Original task description:
        <task_description>
        {task_description}
        </task_description>

        Original task inputs:
        <task_inputs>
        {{ {prompt_inputs} }}
        </task_inputs>

        Solution to Evaluate:
        <solution>
        {output}
        </solution>

        Criteria you should use to evaluate the solution:
        <criteria>
        {solution_criteria}
        </criteria>

        {extra_criteria_section}

        Scoring Guidelines:
        * Score 1-3: Solution fails to meet one or more MANDATORY requirements
        * Score 4-6: Solution meets all mandatory requirements but has significant deficiencies in secondary criteria
        * Score 7-8: Solution meets all mandatory requirements and most secondary criteria, with minor issues
        * Score 9-10: Solution meets all mandatory and secondary criteria

        IMPORTANT SCORING INSTRUCTIONS:
        * Grade the output based ONLY on the listed criteria. Do not add your own extra requirements.
        * If a solution meets all of the mandatory and secondary criteria give it a 10
        * Don't complain that the solution "only" meets the mandatory and secondary criteria. Solutions shouldn't go above and beyond - they should meet the exact listed criteria.
        * ANY violation of a mandatory requirement MUST result in a score of 3 or lower
        * The full 1-10 scale should be utilized - don't hesitate to give low scores when warranted

        Output Format
        Provide your evaluation as a structured JSON object with the following fields, in this specific order:
        - "strengths": An array of 1-3 key strengths
        - "weaknesses": An array of 1-3 key areas for improvement
        - "reasoning": A concise explanation of your overall assessment
        - "score": A number between 1-10

        Respond with JSON. Keep your response concise and direct.
        Example response shape:
        {{
            "strengths": string[],
            "weaknesses": string[],
            "reasoning": string,
            "score": number
        }}
        """

        eval_prompt = self.render(
            dedent(eval_template),
            {
                "task_description": test_case["task_description"],
                "prompt_inputs": prompt_inputs,
                "output": output,
                "solution_criteria": "\n".join(test_case["solution_criteria"]),
                "extra_criteria_section": extra_criteria_section,
            },
        )

        messages = []
        add_messages(messages, eval_prompt, role="user")
        add_messages(messages, "```json", role="assistant")
        eval_text = chat(
            messages,
            stop_sequences=["```"],
            temperature=0.0,
        )
        return json.loads(eval_text)

    def run_test_case(self, test_case, run_prompt_function, extra_criteria=None):
        """Run a test case and grade the result"""
        output = run_prompt_function(test_case["prompt_inputs"])

        model_grade = self.grade_output(test_case, output, extra_criteria)
        model_score = model_grade["score"]
        reasoning = model_grade["reasoning"]

        return {
            "output": output,
            "test_case": test_case,
            "score": model_score,
            "reasoning": reasoning,
        }

    def run_evaluation(
        self,
        run_prompt_function,
        dataset_file,
        extra_criteria=None,
        json_output_file="output.json",
        html_output_file="output.html",
    ):
        """Run evaluation on all test cases in the dataset"""
        with open(dataset_file, "r") as f:
            dataset = json.load(f)

        results = []
        completed = 0
        total = len(dataset)
        last_reported_percentage = 0

        with concurrent.futures.ThreadPoolExecutor(
            max_workers=self.max_concurrent_tasks
        ) as executor:
            future_to_test_case = {
                executor.submit(
                    self.run_test_case,
                    test_case,
                    run_prompt_function,
                    extra_criteria,
                ): test_case
                for test_case in dataset
            }

            for future in concurrent.futures.as_completed(future_to_test_case):
                result = future.result()
                completed += 1
                current_percentage = int((completed / total) * 100)
                milestone_percentage = (current_percentage // 20) * 20

                if milestone_percentage > last_reported_percentage:
                    print(f"Graded {completed}/{total} test cases")
                    last_reported_percentage = milestone_percentage
                results.append(result)

        average_score = mean([result["score"] for result in results])
        print(f"Average score: {average_score}")

        with open(json_output_file, "w") as f:
            json.dump(results, f, indent=2)

        html = generate_prompt_evaluation_report(results)
        with open(html_output_file, "w", encoding="utf-8") as f:
            f.write(html)

        return results

In [27]:
# Creating an instance of PromptEvaluator
# since I have only a basic version of claude api, setting max_concurrent_tasks to avoid any rate limiting issues

evaluator = PromptEvaluator(max_concurrent_tasks=1)

In [15]:
dataset = evaluator.generate_dataset(
    # describing the purpose or goal of the prompt to test out
    task_description="Write a compact, concise 1 day meal plan for a single athlete",
    # listing out and describing the different inputs for the prompt
    prompt_inputs_spec={
        "height": "The athlete's height in centimeters",
        "weight": "The athlete's weight in kilograms",
        "goal": "Goal of the athlete",
        "restrictions": "Dietary restrictions of the athlete"
    },
    # Where to write the generated dataset
    output_file="dataset.json",
    # Number of test cases to generate (recommend keeping this low if you're getting rate limit errors)
    num_cases=3,
)

Generated 1/3 test cases
Generated 2/3 test cases
Generated 3/3 test cases


In [16]:
# Define and run the prompt you want to evaluate, returning the raw model output
# This function is executed once for each test case
def run_prompt(prompt_inputs):
    prompt = f"""
    What should this person eat?

    - Height: {prompt_inputs['height']}
    - Weight: {prompt_inputs['weight']}
    - Goal: {prompt_inputs['goal']}
    - Dietary Restrictions: {prompt_inputs['restrictions']}
    """

    messages = []
    add_messages(messages, prompt, role="user")
    return chat(messages)

In [ ]:
results = evaluator.run_evaluation(
    run_prompt_function=run_prompt, 
    dataset_file="dataset.json",
    extra_criteria="""
    The output should include:
    - Daily caloric total
    - Macronutrient breakdown
    - Meals with exact foods, portions, and timing
    """,
)

# the bare bones simple initial output results in a average score of 3

Graded 1/3 test cases
Graded 2/3 test cases
Graded 3/3 test cases
Average score: 3


### Being clear and direct
Being Clear:
- using simple language
- stating explicitly
- lead the prompt with a simple statement of the models task
- example:
    - instead of: `I need to learn more about renewable energy, I've heard that multiple people have solar panels on their roofs, and apparently theres wind farms just for windmills `
        - (cut out the question, and supporting text for the questions -> focus more on the task at hand, in this case asking about renewable energy)
    - Use: `Write three paragraphs about the types of renewable energy`
        - (the task is stated through an action verb, and the topic is clearly given)

Being Direct:
- use instructions and *not questions*
- use direct action verbs (like: *Write*, *Create*, *Generate*)
- example:
    - Instead of: `I just learned about renewable energy, and it seems like a lot of countries are using it. This sounds like a good idea, what countries use it`
        - (instead of asking a question, just ask for the information)
    - Use: `Identify three counties that are adopting renewable energy. Include generation stats for each`
        - (the topic is outright stated first, with the task stated after as an instruction)


In [18]:
def run_prompt(prompt_inputs):
    prompt = f"""
    Generate a one-day meal plan for an athlete that meets their dietary restrictions

    - Height: {prompt_inputs['height']}
    - Weight: {prompt_inputs['weight']}
    - Goal: {prompt_inputs['goal']}
    - Dietary Restrictions: {prompt_inputs['restrictions']}
    """

    messages = []
    add_messages(messages, prompt, role="user")
    return chat(messages)

In [ ]:
results = evaluator.run_evaluation(
    run_prompt_function=run_prompt, 
    dataset_file="dataset.json",
    extra_criteria="""
    The output should include:
    - Daily caloric total
    - Macronutrient breakdown
    - Meals with exact foods, portions, and timing
    """,
)

# this has improved the score from 2 to 7.67, meaning that the changes made have improved on the score, although there is still bit of room for improvement

Graded 1/3 test cases
Graded 2/3 test cases
Graded 3/3 test cases
Average score: 7.666666666666667


### Being specific 
Listing out guidelines, to direct claude to the output that you're looking for, instead of having claude interpret what you meant. Clear guidelines/steps can be provided to claude for a clearer output

Example:
- instead of: `write a short story about a character who discovers a hidden talent`
    - this story could be 200 or 20,000 words
    - it could have multiple characters
- use: `Write a short short story about a character who discovers a hidden talent. Guidelines: 1. Keep the story under 1,000 words. 2. Include a clear action that reveals the character's talent. 3, Include at least on supporting character`
    - with the *specific* guidelines, claude has a better understanding of what you are looking for

##### Two Types of guidelines
Output Quality: Focusing on qualities that the output should have to control (this is the guideline used for the story example)
- length of the response
- structure and format
- specific attributes or elements to add
- tone or style requirements

Process steps: provides specific steps to follow, useful to get claude to think through a project step by step, or considering multiple options before coming to a final answer
- for the story example:
    - first brainstorming 3 special talents that would make for a good story 
    - then picking the most interesting one, outline
    - think about a interesting scene to reveal that talent
    - think of supporting characters that make the story more interesting

##### When to use each guideline
- always list out qualities that the output should have for almost any prompt
- provide steps for claude when there are:
    - Troubleshooting complex problems
    - Decision-making scenarios
    - Critical thinking tasks
    - Forcing claude to consider a wider view
        - making claude look at and consider other viewpoints and data it might not consider otherwise

In [30]:
def run_prompt(prompt_inputs):
    prompt = f"""
    Generate a one-day meal plan for an athlete that meets their dietary restrictions.

    - Height: {prompt_inputs['height']}
    - Weight: {prompt_inputs['weight']}
    - Goal: {prompt_inputs['goal']}
    - Dietary Restrictions: {prompt_inputs['restrictions']}

    Guidelines:
    1. Include accurate daily caloric amount
    2. Show protein, fat, and carb amounts
    3. Specify when to eat each meal
    4. Use only foods that fit restrictions
    5. List all portion sizes in gram
    6. Keep budget-friendly if mentioned
    """

    messages = []
    add_messages(messages, prompt, role="user")
    return chat(messages)

In [ ]:
results = evaluator.run_evaluation(
    run_prompt_function=run_prompt, 
    dataset_file="dataset.json",
    extra_criteria="""
    The output should include:
    - Daily caloric total
    - Macronutrient breakdown
    - Meals with exact foods, portions, and timing
    """,
)


# with the guidelines also included, the average score jumped from 7.67 to 8
# another major issue/difference is that to save on some tokens, i set it to 500 initially, this let to some of the meal plans being cut off, due to the fewer tokens (by halve)

# when tokens were still set at 500
    # there is an improvement over the previous method, however it is not as significant as the video, since the video has 50 test cases, giving more tries for claude
    # the video also uses the more advanced sonnet model compared to the lowest model haiku I'm using(for both making the dataset, prompts, and evaluation)

Graded 1/3 test cases
Graded 2/3 test cases
Graded 3/3 test cases
Average score: 8


### Structure with XML tags
##### XML Tags
XML tags are markup markers that are defined using angle brackets: <  >. They are used to define data structure, labels and boundaries in markup 
- Tags are case sensitive(`<tag>`, `<Tag>` are different), follow proper nesting(inner tags have to close before outer tags do), and contain a root element(parent root tag that encloses all other tags)
- There are different tags
    - Opening tag for the start of an element: `<tag>`
    - Closing tag for the end of an element using forward slash: `</tag>`
    - Empty tag combining opening and closing tags with no inner text: `<tag \>`

##### How XML tags help with structure 
When building prompts with a lot of content, Claude can sometimes struggle to understand which pieces of text belong to each other
- what different sections represent
- where one section ends and another begins
- XML tags are a way to structure and clear the prompts, especially when large amounts of data and context are included 
    - use XML tags to separate distinct portions of the prompt

Example: Code and Documentation
- if I asked Claude to debug code with provided documentation, adding everything to a single prompt causes confusion
- there is no way for claude to know when the code ends and when documentation begins
- so a better way is to use tags
    - `<my_code>` and `<documentation>` help create clear boundaries for separating code and documentation

##### Custom Tag Names
- there is no need for official XML tags, simply creating descriptive names that make sense for the context
    - `<sales_records>` is better than `<data>`
    - `<athlete_information>` clearly shows users information
    - `<my_code>` and `<documentation>` separate different types of content

#### When to use XML tags
- when there are large amount of varied context or data
- mixing different types of context (code, documentation, data, images, context, etc)
- being clear about content boundaries (where one content ends and another begins)
- working with more complex prompts that include multiple variables

In [28]:
def run_prompt(prompt_inputs):
    # the xml tags: open and closed are put around the information/context for them
    prompt = f"""
    Generate a one-day meal plan for an athlete that meets their dietary restrictions.
    
    <athlete_information>
    - Height: {prompt_inputs['height']}
    - Weight: {prompt_inputs['weight']}
    - Goal: {prompt_inputs['goal']}
    - Dietary Restrictions: {prompt_inputs['restrictions']}
    </athlete_information>

    Guidelines:
    1. Include accurate daily caloric amount
    2. Show protein, fat, and carb amounts
    3. Specify when to eat each meal
    4. Use only foods that fit restrictions
    5. List all portion sizes in gram
    6. Keep budget-friendly if mentioned    
    """

    messages = []
    add_messages(messages, prompt, role="user")
    return chat(messages)

In [ ]:
results = evaluator.run_evaluation(
    run_prompt_function=run_prompt, 
    dataset_file="dataset.json",
    extra_criteria="""
    The output should include:
    - Daily caloric total
    - Macronutrient breakdown
    - Meals with exact foods, portions, and timing
    """,
)

# with the XML tags, there is a drop in performance from 8.67 to 5.53, this could be that since this is a simpler example and the structure did not need to be cleared up using XML tags
# diverging from the course 

Graded 1/3 test cases
Graded 2/3 test cases
Graded 3/3 test cases
Average score: 5.333333333333333


### Providing examples
Is one of the most effective prompt engineering techniques, one-shot or multi-shot prompting
- one example(one-shot) or multiple examples(multi-shot)
- provides sample input/outputs to guide responses

Example: sentiment analysis
- categorizing if a tweet is positive or negative
- the main challenge is detecting sarcasm: saying this was the best movie since `bad movie`

##### Adding Examples to Handle Corner Cases
The improved prompt includes:
- A clear positive example: "Great game tonight!" -> "Positive"
- A sarcastic example: "Oh yeah, I really needed a flight delay tonight! Excellent!" -> "Negative"
- Context explaining why sarcasm should be treated carefully

##### When to Use Examples
Examples are particularly useful for:
- Capturing corner cases or edge scenarios
- Defining complex output formats (like specific JSON structures)
- Showing the exact style or tone you want
- Demonstrating how to handle ambiguous inputs
- multi-shot are used to handle various edge cases or showing diferentent types of valid responses

##### Finding Good Examples from Evaluations
When running prompt evaluations, look for your highest-scoring outputs to use as examples:
- Find responses that scored 10 (or your highest available score) and use those input/output pairs as examples in your prompt
- This helps Claude understand what "perfect" output looks like for your specific use case

##### Best Practices
- Always use XML tags to structure your examples clearly
- Be explicit about what you're showing: "Here is an example input with an ideal response"
- Include examples that address your most common failure cases
- Explain why your example outputs are considered ideal
- Keep examples relevant to your specific task

Practices show rather than tell, instead of describing, it is demonstrated directly 

In [32]:
def run_prompt(prompt_inputs):
    prompt = f"""
    Generate a one-day meal plan for an athlete that meets their dietary restrictions.
    
    <athlete_information>
    - Height: {prompt_inputs['height']}
    - Weight: {prompt_inputs['weight']}
    - Goal: {prompt_inputs['goal']}
    - Dietary Restrictions: {prompt_inputs['restrictions']}
    </athlete_information>

    Guidelines:
    1. Include accurate daily caloric amount
    2. Show protein, fat, and carb amounts
    3. Specify when to eat each meal
    4. Use only foods that fit restrictions
    5. List all portion sizes in gram
    6. Keep budget-friendly if mentioned
    
    Here is an example with a sample input and an ideal output:
    <sample_input>
    height: 180
    weight: 85
    goal: Build muscle mass and strength for weightlifting competition
    restrictions: None
    </sample_input>
    
    <sample_output>
    # One-Day Muscle-Building Meal Plan for Weightlifter
    **Athlete Profile:** 180cm, 85kg male, weightlifting competitor

    ---

    ## Daily Caloric Target
    **3,200 calories** | **220g protein** | **320g carbs** | **90g fat**

    ---

    ## BREAKFAST (7:00 AM)
    *Pre-training fuel*

    - Oatmeal: 80g dry
    - Whole eggs: 3 large (180g)
    - Honey: 20g
    - Banana: 150g
    - Whole milk: 250ml

    **Calories:** 750 | Protein: 28g | Carbs: 85g | Fat: 22g

    ---

    ## PRE-WORKOUT SNACK (10:30 AM)
    *1-2 hours before training*

    - White rice cakes: 60g
    - Peanut butter: 30g
    - Banana: 100g

    **Calories:** 420 | Protein: 12g | Carbs: 52g | Fat: 18g

    ---

    ## LUNCH (1:30 PM)
    *Post-workout meal*

    - Ground beef (90/10): 200g cooked
    - White rice: 250g cooked
    - Broccoli: 150g
    - Olive oil: 10ml

    **Calories:** 820 | Protein: 52g | Carbs: 85g | Fat: 20g

    ---

    ## AFTERNOON SNACK (4:00 PM)

    - Greek yogurt (plain, 0%): 200g
    - Mixed berries (frozen): 100g
    - Granola: 40g
    - Honey: 15g

    **Calories:** 380 | Protein: 25g | Carbs: 52g | Fat: 4g

    ---

    ## DINNER (7:00 PM)

    - Chicken breast: 200g grilled
    - Sweet potato: 250g baked
    - Green beans: 150g
    - Butter: 10g

    **Calories:** 580 | Protein: 55g | Carbs: 55g | Fat: 12g

    ---

    ## EVENING SNACK (9:30 PM)
    *Before bed - slow-digesting protein*

    - Cottage cheese (2%): 150g
    - Almonds: 25g
    - Casein protein powder: 30g (mixed into cottage cheese)

    **Calories:** 270 | Protein: 48g | Carbs: 8g | Fat: 4g

    ---

    ## HYDRATION
    - Water: 3-4 liters throughout day
    - Electrolyte drink during workout: 500ml

    ---

    ## NUTRITIONAL SUMMARY
    | Macro | Amount | Target |
    |-------|--------|--------|
    | **Calories** | 3,200 | ✓ |
    | **Protein** | 220g (27%) | ✓ |
    | **Carbs** | 337g (42%) | ✓ |
    | **Fat** | 80g (23%) | ✓ |

    ---

    ## NOTES FOR SUCCESS

    ✓ **Timing:** Carbs and protein concentrated around workouts (pre/post)  
    ✓ **Budget-Friendly:** Uses affordable staples (eggs, rice, chicken, oats)  
    ✓ **Performance:** 1.1g protein per pound of bodyweight for muscle growth  
    ✓ **Recovery:** Casein at night supports overnight muscle synthesis  
    ✓ **Adjustments:** Add 200 calories if weight gain stalls after 2 weeks
    </sample_output>

    """
    # the sample output that was used was the highest scoring prompt result from the being specific part (the score was a 9/10)

    messages = []
    add_messages(messages, prompt, role="user")
    return chat(messages)

In [33]:
results = evaluator.run_evaluation(
    run_prompt_function=run_prompt, 
    dataset_file="dataset.json",
    extra_criteria="""
    The output should include:
    - Daily caloric total
    - Macronutrient breakdown
    - Meals with exact foods, portions, and timing
    """,
)

Graded 1/3 test cases
Graded 2/3 test cases
Graded 3/3 test cases
Average score: 7.333333333333333


### Exercise on prompting

In [36]:
dataset = evaluator.generate_dataset(
    # Describe the purpose or goal of the prompt you're trying to test
    task_description="""
    Extract topics out of a passage of text from a scholarly article into a JSON array of strings.
    """,
    # Describe the different inputs that your prompt requires
    prompt_inputs_spec={
        "content": "One paragraph of text from a scholarly journal written in English"
    },
    # Where to write the generated dataset
    output_file="dataset.json",
    # Number of test cases to generate (recommend keeping this low if you're getting rate limit errors)
    num_cases=3,
)

Generated 1/3 test cases
Generated 2/3 test cases
Generated 3/3 test cases


In [39]:
def run_prompt(prompt_inputs):
    prompt = f"""
    Extract key topics that were mentioned from a passage of text from a scholarly journal into a JSON array of strings
    
    <text>
    {prompt_inputs["content"]}
    </text>
    
    Follow these steps:
    1. Closely examine the provided text
    2. Identify each topic that was mentioned 
    3. Add the topic to the array of JSON array of strings
    4. Respond with the JSON array. Do not provide any other commentary
    """
    
    # the XML tags were called text, since that the referenced object previously in the prompt

    messages = []
    add_messages(messages, prompt, role="user")
    return chat(messages)

In [ ]:
results = evaluator.run_evaluation(
    run_prompt_function=run_prompt, 
    dataset_file="dataset.json",
    extra_criteria="""
    - Contains a JSON array of strings, containing each topic mentioned in the article.
    - The strings should contain only a topic without any extra commentary or explanation.
    - Response should contain the JSON array and nothing else.
    """,
)

# this results in a score of 7.33

Graded 1/3 test cases
Graded 2/3 test cases
Graded 3/3 test cases
Average score: 7.333333333333333
